In [1]:
import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ventas = pd.read_csv("../datos/crudos/ventas.csv")
print(ventas.shape)
ventas.head(20)

(60770, 10)


,folio,fecha,id_cliente,sku,cantidad,precio_unitario,descuento_pct,ruta,id_vendedor,canal
0,529565,2025-07-04,C-90104,SKU-1104,6,2020.0,10.9,R-02,V-02,Mostrador
1,559138,22/05/2023,C-01624,SKU-1124,12,17090.0,3.7,R-02,V-02,Ruta
2,537107,2025-01-30,C-02339,SKU-1047,24,9030.0,3.0,R-04,V-04,Mostrador
3,525553,31/05/2025,C-00830,SKU-1014,2,8760.0,1.0,R-07,V-07,Ruta
4,514151,08/06/2026,C-01448,SKU-1052,2,5000.0,0.1,R-02,V-02,Mostrador
5,523061,11/05/2026,C-02316,SKU-1004,6,12350.0,3.0,R-04,V-04,Ruta
6,510415,2026-05-03,C-01582,SKU-1060,24,4640.0,9.6,R-02,V-02,Televenta
7,546688,30-11-2023,C-00484,SKU-1149,1,22150.0,1.7,R-02,V-02,Televenta
8,518664,2023-05-08,C-90114,SKU-1097,24,17340.0,5.0,R-04,V-04,Ruta
9,521795,2023-12-29,C-01379,SKU-1077,24,7750.0,35.8,R-05,V-05,Ruta


In [2]:
# 1. Formatos de fecha
formatos = ["%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%m/%d/%Y"]
pendientes = ventas["fecha"].copy()
for f in formatos:
    ok = pd.to_datetime(pendientes, format=f, errors="coerce").notna()
    print(f"{f}: {ok.sum():,} filas")
    pendientes = pendientes[~ok]
print(f"Sin reconocer: {len(pendientes):,}")
print(pendientes.unique()[:10])

# 2, 3. Valores imposibles
print("\nCantidades negativas:", (ventas["cantidad"] < 0).sum())
print("Valores de cantidad:", sorted(ventas["cantidad"].unique()))
print("Precio en cero:", (ventas["precio_unitario"] == 0).sum())

# 5. Integridad referencial
productos = pd.read_csv("../datos/crudos/productos.csv")
clientes = pd.read_excel("../datos/crudos/clientes.xlsx")
print("\nSKU huérfanos:", (~ventas["sku"].isin(productos["sku"])).sum())
print("Clientes huérfanos:", (~ventas["id_cliente"].isin(clientes["id_cliente"])).sum())

%Y-%m-%d: 33,206 filas
%d/%m/%Y: 16,895 filas
%d-%m-%Y: 7,974 filas
%m/%d/%Y: 2,604 filas
Sin reconocer: 91
<StringArray>
['2027-13-45']
Length: 1, dtype: str

Cantidades negativas: 403
Valores de cantidad: [np.int64(-24), np.int64(-12), np.int64(-6), np.int64(-4), np.int64(-3), np.int64(-2), np.int64(-1), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(12), np.int64(24)]
Precio en cero: 250

SKU huérfanos: 122
Clientes huérfanos: 181


In [3]:
print("SKUs huérfanos distintos:",
      ventas.loc[~ventas["sku"].isin(productos["sku"]), "sku"].unique())
print("Clientes huérfanos distintos:",
      ventas.loc[~ventas["id_cliente"].isin(clientes["id_cliente"]), "id_cliente"].nunique())
print("\nSKUs en catálogo que nunca se vendieron:",
      (~productos["sku"].isin(ventas["sku"])).sum())

SKUs huérfanos distintos: <StringArray>
['SKU-0000']
Length: 1, dtype: str
Clientes huérfanos distintos: 1

SKUs en catálogo que nunca se vendieron: 0


In [4]:
print("SKUs huérfanos distintos:",
      ventas.loc[~ventas["sku"].isin(productos["sku"]), "sku"].unique())
print("Clientes huérfanos distintos:",
      ventas.loc[~ventas["id_cliente"].isin(clientes["id_cliente"]), "id_cliente"].unique())
print("SKUs en catálogo que nunca se vendieron:",
      (~productos["sku"].isin(ventas["sku"])).sum())
print("Rango de fechas válidas:")
f = pd.to_datetime(ventas["fecha"], format="mixed", dayfirst=True, errors="coerce")
print(f.min(), "a", f.max())

SKUs huérfanos distintos: <StringArray>
['SKU-0000']
Length: 1, dtype: str
Clientes huérfanos distintos: <StringArray>
['C-99999']
Length: 1, dtype: str
SKUs en catálogo que nunca se vendieron: 0
Rango de fechas válidas:
2023-01-01 00:00:00 a 2026-12-06 00:00:00


In [5]:
import json
devoluciones = pd.read_csv("../datos/crudos/devoluciones.csv")
vendedores   = pd.read_excel("../datos/crudos/vendedores.xlsx")
with open("../datos/crudos/visitas_ruta.json", encoding="utf-8") as fh:
    visitas = pd.json_normalize(json.load(fh))

for nombre, df in [("CLIENTES", clientes), ("PRODUCTOS", productos),
                   ("DEVOLUCIONES", devoluciones), ("VENDEDORES", vendedores),
                   ("VISITAS", visitas)]:
    display(perfilar(df, nombre))

NameError: name 'perfilar' is not defined

In [ ]:
print("Duplicados exactos:", clientes.duplicated().sum())
print("id_cliente repetidos:", clientes["id_cliente"].duplicated().sum())
print("Ciudades distintas:", sorted(clientes["ciudad"].unique()))
print("Formatos de NIT:", clientes["nit"].head(15).tolist())
print("Correos con más de un @:", clientes["email"].astype(str).str.count("@").gt(1).sum())

In [ ]:
print(productos["categoria"].value_counts())
print("Costo >= precio:",
      (productos["costo_unitario"] >= productos["precio_lista"]).sum())

In [ ]:
## Conclusión del perfilado

De 60.770 registros de ventas, 600 son duplicados exactos y 91 tienen fecha imposible: 691 se descartan con motivo registrado. 403 corresponden a devoluciones cargadas como cantidad negativa y se reclasifican, no se eliminan. 250 con precio en cero quedan pendientes de decisión tras cruzar contra el catálogo. 122 registros apuntan a SKUs inexistentes y 181 a clientes inexistentes; se marcan y se reportan al cliente. La base útil estimada es de 60079 registros.  La base útil estimada es de 60.079 registros, de los cuales 403 se reclasifican como devoluciones y 303 quedan marcados por integridad referencial pendiente. ####

In [6]:
descartadas = 600 + 91          # duplicados exactos + fechas imposibles
print("Base útil estimada:", len(ventas) - descartadas)

Base útil estimada: 60079


In [7]:
ventas = ventas.drop_duplicates()

In [8]:
ventas = ventas.drop_duplicates()   # siempre primero

def clasificar_fecha(texto):
    if "-" in texto and len(texto.split("-")[0]) == 4:
        # ¿es ISO válida o inválida? (pista: pd.to_datetime con errors="coerce")
        ...
    elif "-" in texto:
        return "dd-mm-aaaa"
    else:
        a, b, anio = texto.split("/")
        a, b = int(a), int(b)
        # aquí van tus tres casos
        ...

ventas["tipo_fecha"] = ventas["fecha"].apply(clasificar_fecha)
print(ventas["tipo_fecha"].value_counts())
print("Total:", ventas["tipo_fecha"].value_counts().sum())

tipo_fecha
dd-mm-aaaa    7880
Name: count, dtype: int64
Total: 7880


In [9]:
print(ventas["tipo_fecha"].value_counts(dropna=False))

tipo_fecha
NaN           52290
dd-mm-aaaa     7880
Name: count, dtype: int64


In [10]:
import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ventas = pd.read_csv("../datos/crudos/ventas.csv")
print(ventas.shape)

(60770, 10)


In [11]:
antes = len(ventas)
ventas = ventas.drop_duplicates()
print("Duplicados eliminados:", antes - len(ventas))
print("Filas restantes:", len(ventas))

Duplicados eliminados: 600
Filas restantes: 60170


In [12]:
ventas_crudo = pd.read_csv("../datos/crudos/ventas.csv")
perfilar(ventas_crudo)

NameError: name 'perfilar' is not defined

In [13]:
print(ventas_crudo.groupby("id_vendedor")["ruta"].nunique())

id_vendedor
V-01    1
V-02    1
V-03    1
V-04    1
V-05    1
V-06    1
V-07    1
V-08    1
Name: ruta, dtype: int64


In [14]:
print(ventas_crudo.groupby("ruta")["id_vendedor"].nunique())

ruta
R-01    1
R-02    1
R-03    1
R-04    1
R-05    1
R-06    1
R-07    1
R-08    1
Name: id_vendedor, dtype: int64


In [15]:
import json

productos    = pd.read_csv("../datos/crudos/productos.csv")
devoluciones = pd.read_csv("../datos/crudos/devoluciones.csv")
clientes     = pd.read_excel("../datos/crudos/clientes.xlsx")
vendedores   = pd.read_excel("../datos/crudos/vendedores.xlsx")

with open("../datos/crudos/visitas_ruta.json", encoding="utf-8") as f:
    visitas = pd.json_normalize(json.load(f))

for nombre, df in [("productos", productos), ("devoluciones", devoluciones),
                   ("clientes", clientes), ("vendedores", vendedores),
                   ("visitas", visitas)]:
    print(f"\n===== {nombre}: {df.shape[0]:,} filas x {df.shape[1]} columnas =====")
    display(perfilar(df))


===== productos: 152 filas x 6 columnas =====


NameError: name 'perfilar' is not defined

In [38]:
for tipo in ["barra: día/mes seguro", "barra: mes/día seguro", "barra: ambigua"]:
    ejemplos = ventas.loc[ventas["tipo_fecha"] == tipo, "fecha"].head(5).tolist()
    print(tipo, "→", ejemplos)

KeyError: 'tipo_fecha'

In [39]:
conteo = ventas["tipo_fecha"].value_counts()

ambiguas = conteo["barra: ambigua"]
base_util = len(ventas) - conteo["inválida"]
print(f"Ambiguas: {ambiguas:,} = {ambiguas / base_util:.1%} de la base útil")

dm = conteo["barra: día/mes seguro"]
md = conteo["barra: mes/día seguro"]
print(f"Entre las seguras: día/mes {dm / (dm + md):.0%} | mes/día {md / (dm + md):.0%}")

KeyError: 'tipo_fecha'

In [17]:
import os
print("Estoy en:", os.getcwd())
print("¿Existe la carpeta scripts?", os.path.exists("../scripts"))
if os.path.exists("../scripts"):
    print("Contenido de scripts:", os.listdir("../scripts"))

Estoy en: C:\Users\Asus-pc\andina\notebooks
¿Existe la carpeta scripts? True
Contenido de scripts: ['.ipynb_checkpoints', '02_limpieza.py', 'herramientas_perfilado.py', '__pycache__']


In [18]:
import os
print(os.listdir("../scripts"))

['.ipynb_checkpoints', '02_limpieza.py', 'herramientas_perfilado.py', '__pycache__']


In [19]:
print(os.listdir("../scripts"))

['.ipynb_checkpoints', '02_limpieza.py', 'herramientas_perfilado.py', '__pycache__']


In [20]:
import glob, os, shutil

candidatos = glob.glob(os.path.expanduser("~/Downloads/herramientas_perfilado*"))
print("Encontrados:", candidatos)

if candidatos:
    shutil.copy(candidatos[0], "../scripts/herramientas_perfilado.py")
    print("Copiado. Contenido de scripts:", os.listdir("../scripts"))

Encontrados: []


In [21]:
import os, sys
print(os.listdir("../scripts"))

sys.path.append("../scripts")
from herramientas_perfilado import perfilar, patrones, huerfanos, clasificar_fecha
print("Importado correctamente")

['.ipynb_checkpoints', '02_limpieza.py', 'herramientas_perfilado.py', '__pycache__']
Importado correctamente


In [22]:
import pandas as pd
import os, sys
sys.path.append("../scripts")
from herramientas_perfilado import perfilar, patrones, huerfanos, clasificar_fecha

In [23]:
import json

productos    = pd.read_csv("../datos/crudos/productos.csv")
devoluciones = pd.read_csv("../datos/crudos/devoluciones.csv")
clientes     = pd.read_excel("../datos/crudos/clientes.xlsx")
vendedores   = pd.read_excel("../datos/crudos/vendedores.xlsx")

with open("../datos/crudos/visitas_ruta.json", encoding="utf-8") as f:
    visitas = pd.json_normalize(json.load(f))

for nombre, df in [("productos", productos), ("devoluciones", devoluciones),
                   ("clientes", clientes), ("vendedores", vendedores),
                   ("visitas", visitas)]:
    print(f"\n===== {nombre}: {df.shape[0]:,} filas x {df.shape[1]} columnas =====")
    display(perfilar(df))


===== productos: 152 filas x 6 columnas =====


,columna,tipo,nulos,% nulos,valores_unicos,ejemplo,min,max
0,sku,str,0,0.0,152,SKU-1000,NaN,NaN
1,descripcion,str,0,0.0,152,BEB Producto 1 x250ml,NaN,NaN
2,categoria,str,0,0.0,19,BEBIDAS,NaN,NaN
3,costo_unitario,float64,0,0.0,140,17100.0,0.0,28600.5
4,precio_lista,float64,6,3.9,143,21380.0,1070.0,25530.0
5,proveedor,str,0,0.0,12,PROV-09,NaN,NaN



===== devoluciones: 3,200 filas x 6 columnas =====


,columna,tipo,nulos,% nulos,valores_unicos,ejemplo,min,max
0,id_devolucion,str,0,0.0,3200,DEV-00001,NaN,NaN
1,fecha,str,0,0.0,2064,01/20/2025,NaN,NaN
2,id_cliente,str,0,0.0,1807,C-01399,NaN,NaN
3,sku,str,0,0.0,152,SKU-1076,NaN,NaN
4,cantidad,int64,0,0.0,8,4,1.0,8.0
5,motivo,str,0,0.0,7,Vencido,NaN,NaN



===== clientes: 2,620 filas x 11 columnas =====


,columna,tipo,nulos,% nulos,valores_unicos,ejemplo,min,max
0,id_cliente,str,0,0.0,2580,C-01700,NaN,NaN
1,nit,str,0,0.0,2400,800 101 699,NaN,NaN
2,razon_social,str,0,0.0,1201,PANADERÍA SAN RAFAEL LTDA.,NaN,NaN
3,tipo_negocio,str,0,0.0,6,Panadería,NaN,NaN
4,ciudad,str,0,0.0,24,Santa Rosa de Cabal,NaN,NaN
5,direccion,str,0,0.0,2394,Cra 21 # 15-96,NaN,NaN
6,telefono,str,310,11.8,2274,+57 3151208890,NaN,NaN
7,email,str,75,2.9,1590,san34@outlook.com,NaN,NaN
8,ruta_asignada,str,0,0.0,8,R-03,NaN,NaN
9,fecha_alta,str,0,0.0,1951,05/11/2021,NaN,NaN



===== vendedores: 8 filas x 6 columnas =====


,columna,tipo,nulos,% nulos,valores_unicos,ejemplo,min,max
0,id_vendedor,str,0,0.0,8,V-01,NaN,NaN
1,nombre,str,0,0.0,8,Marcela Betancur,NaN,NaN
2,ruta,str,0,0.0,8,R-01,NaN,NaN
3,ciudad_base,str,0,0.0,5,Pereira,NaN,NaN
4,fecha_ingreso,str,0,0.0,8,2022-09-02,NaN,NaN
5,salario_mensual,int64,0,0.0,3,1600000,1450000.0,1750000.0



===== visitas: 14,000 filas x 10 columnas =====


,columna,tipo,nulos,% nulos,valores_unicos,ejemplo,min,max
0,visita_id,str,0,0.0,14000,VIS-000001,NaN,NaN
1,fecha,str,0,0.0,1277,2026-01-18,NaN,NaN
2,vendedor.id,str,0,0.0,8,V-05,NaN,NaN
3,vendedor.ruta,str,0,0.0,8,R-04,NaN,NaN
4,cliente.id,str,0,0.0,2575,C-01188,NaN,NaN
5,resultado.efectiva,bool,0,0.0,2,True,False,True
6,resultado.motivo_no_venta,str,9784,69.9,4,Dueño ausente,NaN,NaN
7,resultado.duracion_min,int64,0,0.0,42,30,4,45
8,geo.lat,float64,0,0.0,13209,4.32405,4.2002,5.39995
9,geo.lon,float64,0,0.0,13092,-75.25443,-76.19993,-75.20013


In [24]:
# A. ¿Cuántas categorías reales hay?
productos["categoria"].value_counts()

categoria
BEBIDAS       15
Snacks        12
Bebida        11
abarrote      10
Snaks          9
Aseo Hogar     9
aseo           9
ABARROTES      9
Abarrotes      9
snack          8
Lacteos        8
Lácteos        8
ASEO           7
lacteos        7
LACTEOS        7
bebidas        4
Aseo           4
Bebidas        3
SNACKS         3
Name: count, dtype: int64

In [25]:
# B. ¿Cuántos motivos de devolución reales hay?
devoluciones["motivo"].value_counts()

motivo
Vencido               486
No solicitado         462
averiado              461
Producto averiado     457
Error de despacho     451
PRODUCTO AVERIADO     449
Empaque defectuoso    434
Name: count, dtype: int64

In [26]:
# C. ¿Los vacíos de motivo corresponden a visitas efectivas?
pd.crosstab(visitas["resultado.efectiva"], visitas["resultado.motivo_no_venta"].isna())

resultado.motivo_no_venta,False,True
resultado.efectiva,,
False,4216,1075
True,0,8709


In [27]:
# D. ¿En visitas cada vendedor tiene una sola ruta, como en ventas?
visitas.groupby("vendedor.id")["vendedor.ruta"].nunique()

vendedor.id
V-01    8
V-02    8
V-03    8
V-04    8
V-05    8
V-06    8
V-07    8
V-08    8
Name: vendedor.ruta, dtype: int64

In [28]:
(productos["costo_unitario"] == 0).sum()

np.int64(4)

In [29]:
v = pd.read_csv("../datos/procesados/ventas_limpias.csv", parse_dates=["fecha"])
fuera = v[v["fecha"] > "2026-06-30"]
print(len(fuera), "ventas después de junio 2026")
print(fuera["fecha_ambigua"].value_counts())

0 ventas después de junio 2026
Series([], Name: count, dtype: int64)


In [30]:
dev = pd.read_csv("../datos/procesados/devoluciones_desde_ventas.csv")
print("Precio en cero:", (dev["precio_unitario"] == 0).sum())
print("SKU comodín:", (dev["sku"] == "SKU-0000").sum())
print("Cliente comodín:", (dev["id_cliente"] == "C-99999").sum())

Precio en cero: 1
SKU comodín: 2
Cliente comodín: 3


In [31]:
clientes = pd.read_excel("../datos/crudos/clientes.xlsx")
clientes["ciudad"].value_counts()

ciudad
Sta Rosa de Cabal      144
La Virginia            138
SANTA ROSA             134
Manizales              132
Santa Rosa de Cabal    130
LA VIRGINIA            130
Armenia                125
Armenía                123
Lavirginia             122
ARMENIA                121
Cartago Valle          118
Dosquebradas           115
Manizalez              114
MANIZALES              109
Cartago                107
CARTAGO                106
Dos Quebradas          101
DOSQUEBRADAS            99
Dosquebrdas             91
pereira                 85
Pereria                 75
Pereira                 68
Perera                  67
PEREIRA                 66
Name: count, dtype: int64

In [32]:
CIUDADES = {
    "Sta Rosa de Cabal": "Santa Rosa de Cabal",
    "SANTA ROSA": "Santa Rosa de Cabal",
    # ... completa las 24
}

In [33]:
print(clientes["ciudad"].map(CIUDADES).nunique())

1


In [34]:
faltan = set(clientes["ciudad"]) - set(CIUDADES)
print(len(faltan), "escrituras sin mapear:")
print(faltan)

22 escrituras sin mapear:
{'PEREIRA', 'Cartago', 'MANIZALES', 'Manizalez', 'Lavirginia', 'Santa Rosa de Cabal', 'Cartago Valle', 'CARTAGO', 'Pereria', 'Manizales', 'Pereira', 'Armenia', 'pereira', 'Armenía', 'LA VIRGINIA', 'Dosquebrdas', 'Dos Quebradas', 'Perera', 'Dosquebradas', 'ARMENIA', 'La Virginia', 'DOSQUEBRADAS'}


In [35]:
print(clientes["ciudad"].map(CIUDADES).nunique())

1


In [36]:
CIUDADES = {
    # Pereira
    "Pereira": "Pereira",
    "PEREIRA": "Pereira",
    "pereira": "Pereira",
    "Pereria": "Pereira",
    "Perera": "Pereira",

    # Santa Rosa de Cabal
    "Santa Rosa de Cabal": "Santa Rosa de Cabal",
    "Sta Rosa de Cabal": "Santa Rosa de Cabal",
    "SANTA ROSA": "Santa Rosa de Cabal",

    # Dosquebradas
        
    "Dosquebradas": "Dosquebradas",
    "DOSQUEBRADAS": "Dosquebradas",
    "Dosquebrdas": "Dosquebradas",
    "Dos Quebradas": "Dosquebradas",

    # Manizales
    "Manizales": "Manizales",
    "Manizalez": "Manizales",
    "MANIZALES": "Manizales",

    # Armenia
    "ARMENIA": "Armenia",
    "Armenía": "Armenia",
    "Armenia": "Armenia",

    # Cartago
    "CARTAGO": "Cartago",
    "Cartago": "Cartago",
    "Cartago Valle": "Cartago",

    # La Virginia
    "Lavirginia": "La Virginia",
    "La Virginia": "La Virginia",
    "LA VIRGINIA": "La Virginia",
}

faltan = set(clientes["ciudad"]) - set(CIUDADES)
print(len(faltan), "escrituras sin mapear:", faltan)
print("Ciudades finales:", clientes["ciudad"].map(CIUDADES).nunique())

0 escrituras sin mapear: set()
Ciudades finales: 7


In [41]:
clientes["ciudad"].map(CIUDADES).value_counts()

ciudad
Santa Rosa de Cabal    408
Dosquebradas           406
La Virginia            390
Armenia                369
Pereira                361
Manizales              355
Cartago                331
Name: count, dtype: int64

# Orden	Criterio	Por qué
1	Mismo NIT normalizado = mismo cliente	El NIT identifica a la empresa; el nombre no (hay 1.201 razones sociales para 2.400 NIT)
2	Conservar el que tenga más campos llenos (teléfono, email, dirección)	El derrotero pide conservar el más completo
3	Si empatan: el que tenga más ventas asociadas	Es el código que el negocio usa (tu decisión)
4	Si siguen empatados: el id_cliente menor	Garantiza el mismo resultado en cada corrida
5	Las ventas, devoluciones y visitas del registro descartado se reasignan al conservado	Ningún movimiento queda huérfano
6	Cada registro descartado va a descartados.csv con el motivo "Duplicado lógico de C-xxxxx"	Trazabilidad

# DEVOLUCIONES#

In [44]:
devoluciones = pd.read_csv("../datos/crudos/devoluciones.csv")
devoluciones["motivo"].value_counts()

motivo
Vencido               486
No solicitado         462
averiado              461
Producto averiado     457
Error de despacho     451
PRODUCTO AVERIADO     449
Empaque defectuoso    434
Name: count, dtype: int64

In [47]:
MOTIVOS = {
    "PRODUCTO AVERIADO": "Producto averiado",
    "Error de Despacho": "Error de despacho",
    # ... completa las 24
}

In [48]:
print(devoluciones["motivo"].map(MOTIVOS).nunique())

1


In [49]:
faltan = set(devoluciones["motivo"]) - set(MOTIVOS)
print(len(faltan), "escrituras sin mapear:")
print(faltan)

6 escrituras sin mapear:
{'No solicitado', 'Producto averiado', 'averiado', 'Error de despacho', 'Vencido', 'Empaque defectuoso'}


In [51]:
MOTIVOS = {
    "No solicitado": "No solicitado",
    "averiado": "Producto averiado",
    "Error de despacho": "Error de despacho",
    "vencido": "Vencido",
    "Empaque defectuoso": "Empaque defectuoso",
    "Vencido": "Vencido",                       # con V mayúscula, como en los datos
    "Producto averiado": "Producto averiado",
    "PRODUCTO AVERIADO": "Producto averiado",
    
}
faltan = set(devoluciones["motivo"]) - set(MOTIVOS)
print(len(faltan), "escrituras sin mapear:", faltan)
print("Devoluciones finales:", devoluciones["motivo"].map(MOTIVOS).nunique())

0 escrituras sin mapear: set()
Devoluciones finales: 5


In [53]:
from pathlib import Path
import json
import pandas as pd
from herramientas_perfilado import clasificar_fecha